# Fusion Perception — Pipeline → Fine-tune Gemma 4 2B

End-to-end notebook:
1. **Sections 1–2** — set up environment, run the perception pipeline on KITTI, collect `reasoning.json` outputs
2. **Section 3** — extract and inspect the training dataset
3. **Section 4** — fine-tune Gemma 4 2B with QLoRA (HuggingFace PEFT, no Unsloth)
4. **Section 5** — evaluate the fine-tuned adapter

**Before running:** Runtime → Change runtime type → **T4 GPU**

## Section 1 — Environment

In [ ]:
# Cell 1 — Mount Google Drive and navigate to project
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/Fusion_sensor'   # ← edit if needed
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 2 — Install all dependencies
# PyTorch with CUDA 12.1 (T4)
!pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 \
    --index-url https://download.pytorch.org/whl/cu121 -q

# Project + fine-tuning deps
!pip install \
    "omegaconf>=2.3.0" "opencv-python>=4.9.0" "Pillow>=10.0.0" \
    "numpy>=1.26.0" "dataclasses-json>=0.6.0" "rich>=13.0.0" \
    "huggingface_hub>=0.26.0" "transformers>=4.47.0" \
    "bitsandbytes>=0.44.0" "accelerate>=1.2.0" \
    "peft>=0.14.0" "trl>=0.12.0" "datasets>=3.0.0" \
    "matplotlib>=3.8.0" -q

# External model libraries
!pip install git+https://github.com/allenai/WildDet3D -q
!pip install git+https://github.com/facebookresearch/co-tracker -q

# Install local package
!pip install -e . -q

print('Installation complete.')

In [ ]:
# Cell 3 — Verify GPU
import torch

assert torch.cuda.is_available(), 'No GPU — switch runtime to T4'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {vram_gb:.1f} GB')
assert vram_gb >= 10.0, f'Need >= 10 GB VRAM, got {vram_gb:.1f} GB'
print('GPU check passed')

## Section 2 — Run Perception Pipeline

Runs the full stack (WildDet3D → CoWTracker → BEV → Gemma reasoning) on the KITTI
`2011_10_03_drive_0047` sequence. Each Gemma call saves `prompt_used` + `summary`
to `reasoning.json` — this becomes the fine-tuning dataset.

In [ ]:
# Cell 4 — Pipeline configuration
import os, datetime
import numpy as np
from pathlib import Path

DATA_PATH_ROOT = '2011_10_03'
DRIVE_SEQ      = '2011_10_03_drive_0047_sync'
VIDEO_PATH     = '/content/kitti_0047.mp4'
MAX_FRAMES     = 300    # ~20 s at 10 fps; set None for full sequence

# Reasoning fires every N frames — more frequent = more training examples
REASONING_INTERVAL = 15   # default config is 30; halve it here for more data

run_id  = datetime.datetime.now().strftime('%Y-%m-%d_%H%M%S') + '_kitti_0047'
out_dir = Path('outputs') / run_id
out_dir.mkdir(parents=True, exist_ok=True)
print(f'Run ID: {run_id}')
print(f'Output: {out_dir}')

In [ ]:
# Cell 5 — Download WildDet3D checkpoint
!mkdir -p ckpt
!huggingface-cli download allenai/WildDet3D \
    wilddet3d_alldata_all_prompt_v1.0.pt --local-dir ckpt/

In [ ]:
# Cell 6 — Download KITTI raw data (skipped if already on Drive)
import os
seq_dir = os.path.join(DATA_PATH_ROOT, DRIVE_SEQ)
if not os.path.exists(seq_dir):
    !wget -q https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_10_03_drive_0047/2011_10_03_drive_0047_sync.zip
    !wget -q https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_10_03_calib.zip
    !jar xf 2011_10_03_drive_0047_sync.zip
    !jar xf 2011_10_03_calib.zip
    print('Download complete.')
else:
    print('Data already present — skipping download.')

In [ ]:
# Cell 7 — Parse KITTI calibration → real 3×3 intrinsics K
with open(os.path.join(DATA_PATH_ROOT, 'calib_cam_to_cam.txt')) as f:
    calib = f.readlines()

P_rect = np.array(
    [float(x) for x in calib[25].strip().split(' ')[1:]]
).reshape(3, 4)
K = P_rect[:3, :3].astype(np.float32)

print('K matrix (3×3 intrinsics):')
print(K)
print(f'  fx={K[0,0]:.2f}  fy={K[1,1]:.2f}  cx={K[0,2]:.2f}  cy={K[1,2]:.2f}')

In [ ]:
# Cell 8 — Collect image paths and compute real FPS
import pandas as pd
from glob import glob

left_imgs = sorted(glob(os.path.join(DATA_PATH_ROOT, DRIVE_SEQ, 'image_02/data/*.png')))
print(f'Total frames: {len(left_imgs)}')

ts = pd.read_csv(
    os.path.join(DATA_PATH_ROOT, DRIVE_SEQ, 'image_02/timestamps.txt'),
    header=None
).squeeze().astype(object).apply(lambda x: x.split(' ')[1])

def hms(t):
    h, m, s = t.split(':')
    return int(h)*3600 + int(m)*60 + float(s)

secs      = ts.apply(hms)
KITTI_FPS = float(1.0 / np.median(np.diff(secs.values)))
print(f'FPS: {KITTI_FPS:.2f}')

In [ ]:
# Cell 9 — Encode KITTI frames → MP4
import cv2

paths = left_imgs if MAX_FRAMES is None else left_imgs[:MAX_FRAMES]
sample = cv2.imread(paths[0])
H, W = sample.shape[:2]

writer = cv2.VideoWriter(VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), KITTI_FPS, (W, H))
for p in paths:
    writer.write(cv2.imread(p))
writer.release()
print(f'Encoded {len(paths)} frames → {VIDEO_PATH}  ({W}×{H} @ {KITTI_FPS:.1f} fps)')

In [ ]:
# Cell 10 — Smoke test (CPU only, verifies imports)
from fusion_perception.utils.dataclasses import Track, OccupancyGrid, SceneMemory
from fusion_perception.utils.logging_setup import setup_logging, get_logger
from fusion_perception.occupancy.bev_grid import OccupancyBEVGenerator
from fusion_perception.reasoning.prompt_builder import build_scene_prompt

setup_logging(level='INFO', use_rich=False)
logger = get_logger('smoke')

bev  = OccupancyBEVGenerator(resolution=0.5, x_range=[-20.,20.], z_range=[0.,50.], decay_factor=0.95)
grid = bev.update([], frame_idx=0)
assert len(grid.grid) == 100

track = Track(
    track_id=1, class_name='car', first_seen=0, last_seen=5,
    centroid_history=[[320.,240.]], position_3d_history=[[0.,0.,15.]],
    cow_query_point=[320.,240.], is_active=True, occlusion_count=0,
)
occ = OccupancyGrid(frame_idx=0, resolution=0.5, x_range=[-20.,20.], z_range=[0.,50.],
                    grid=[[0.]*80 for _ in range(100)], decay_factor=0.95)
mem = SceneMemory(frame_idx=0, active_tracks=[track], occupancy_grid=occ,
                  event_flags=[], frame_count=1, elapsed_seconds=0.0)
prompt = build_scene_prompt(mem)
assert 'car' in prompt and K.shape == (3, 3)
print('Smoke test passed')

In [ ]:
# Cell 11 — Run perception pipeline
#
# Reasoning fires every REASONING_INTERVAL frames.
# Each call saves prompt_used + summary to reasoning.json.
# With 300 frames at interval=15 you get ~20 training examples.
# More frames = more data = better fine-tuning.

from omegaconf import OmegaConf
from fusion_perception.models.wilddet3d_wrapper import WildDet3DWrapper
from fusion_perception.tracking.kalman_cowtracker import KalmanCoWTracker
from fusion_perception.occupancy.bev_grid import OccupancyBEVGenerator
from fusion_perception.tracking.trajectory_manager import SceneMemoryManager
from fusion_perception.models.gemma_wrapper import GemmaReasoningWrapper
from fusion_perception.pipelines.stage_runner import StageRunner
from fusion_perception.utils.video_loader import VideoLoader
from fusion_perception.utils.json_io import save_detections, save_tracks, save_reasoning
from fusion_perception.utils.memory_monitor import log_gpu_memory

cfg = OmegaConf.merge(
    OmegaConf.load('configs/default.yaml'),
    OmegaConf.load('configs/colab.yaml'),
)

detector = WildDet3DWrapper(score_threshold=cfg.detection.score_threshold, fp16=cfg.detection.fp16)
detector.load(cfg.detection.checkpoint, cfg.detection.device)

tracker = KalmanCoWTracker(
    window_size=cfg.tracking.window_size, max_tracks=cfg.tracking.max_tracks,
    lost_patience=cfg.tracking.lost_patience, confirm_age=cfg.tracking.confirm_age,
    high_score_threshold=cfg.tracking.high_score_threshold,
    low_score_threshold=cfg.tracking.low_score_threshold,
    assignment_cost_threshold=cfg.tracking.assignment_cost_threshold,
    alpha_cost=cfg.tracking.alpha_cost, cow_conf_threshold=cfg.tracking.cow_conf_threshold,
    min_cow_points=cfg.tracking.min_cow_points, velocity_decay=cfg.tracking.velocity_decay,
    ego_motion=cfg.tracking.ego_motion, mahal_threshold=cfg.tracking.mahal_threshold,
    lazy_cow_innovation=cfg.tracking.lazy_cow_innovation, device=cfg.tracking.device,
)
tracker.load()

bev_gen   = OccupancyBEVGenerator(resolution=cfg.occupancy.resolution,
                                   x_range=list(cfg.occupancy.x_range),
                                   z_range=list(cfg.occupancy.z_range),
                                   decay_factor=cfg.occupancy.decay_factor,
                                   lidar_confidence=cfg.occupancy.lidar_confidence)
scene_mem = SceneMemoryManager()
gemma     = GemmaReasoningWrapper(model_id=cfg.reasoning.model_id,
                                   quantize_4bit=cfg.reasoning.quantize_4bit,
                                   max_new_tokens=cfg.reasoning.max_new_tokens,
                                   device=cfg.reasoning.device,
                                   visual_mode=False)
gemma.load()
log_gpu_memory('All models loaded')

runner = StageRunner(
    detector=detector, tracker=tracker, bev_generator=bev_gen,
    scene_memory=scene_mem, gemma=gemma,
    prompts=list(cfg.detection.prompts),
    reasoning_interval=REASONING_INTERVAL,
    fps=KITTI_FPS, visual_reasoning=False,
)

loader = VideoLoader(source=VIDEO_PATH, resize_hw=None, max_frames=MAX_FRAMES)
out_video = cv2.VideoWriter(str(out_dir / 'output.mp4'),
                             cv2.VideoWriter_fourcc(*'mp4v'),
                             KITTI_FPS, (W + H + 200, H))

all_detections, all_tracks, all_reasoning = {}, {}, []
FLUSH = cfg.output.flush_interval

for frame_idx, frame_rgb, meta in loader:
    outputs = runner.run_frame(frame_rgb, frame_idx, intrinsics=K)
    all_detections[frame_idx] = outputs['detections']
    for t in outputs['tracks']:
        all_tracks[t.track_id] = t
    if outputs['reasoning']:
        r = outputs['reasoning']
        all_reasoning.append(r)
        print(f'  Frame {frame_idx:4d} | reasoning: {r.summary[:80]}')
    out_video.write(outputs['composite'][:, :, ::-1])
    if (frame_idx + 1) % FLUSH == 0:
        save_reasoning(out_dir / 'reasoning.json', run_id, REASONING_INTERVAL, all_reasoning)

save_detections(out_dir / 'detections.json', run_id, VIDEO_PATH,
                list(cfg.detection.prompts), all_detections)
save_tracks(out_dir / 'tracks.json', run_id, all_tracks)
save_reasoning(out_dir / 'reasoning.json', run_id, REASONING_INTERVAL, all_reasoning)
out_video.release()

print(f'\nDone. {frame_idx+1} frames | {len(all_reasoning)} reasoning outputs → {out_dir}')

In [ ]:
# Cell 12 — Preview annotated output frame
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (22, 6)

cap = cv2.VideoCapture(str(out_dir / 'output.mp4'))
cap.set(cv2.CAP_PROP_POS_FRAMES, min(30, (frame_idx // 2)))
ret, bgr = cap.read()
cap.release()

if ret:
    plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    plt.title('Pipeline output — annotated frame | BEV grid | reasoning panel')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Could not read output video.')

## Section 3 — Training Dataset

Every `reasoning.json` already contains `prompt_used` (the scene prompt sent to Gemma)
and `summary` (the model's response). The cell below collects all outputs from every
pipeline run, filters short entries, prints examples, and writes `data/finetune/train.jsonl`.

In [ ]:
# Cell 13 — Extract training data from all reasoning.json outputs
import json
from pathlib import Path
import matplotlib.pyplot as plt

OUTPUTS_DIR     = Path('outputs')
OUT_JSONL       = Path('data/finetune/train.jsonl')
MIN_PROMPT_LEN  = 30
MIN_SUMMARY_LEN = 20

OUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

examples, skipped = [], 0
for rf in sorted(OUTPUTS_DIR.rglob('reasoning.json')):
    with open(rf) as f:
        data = json.load(f)
    for entry in data.get('outputs', []):
        prompt  = entry.get('prompt_used', '').strip()
        summary = entry.get('summary', '').strip()
        if len(prompt) < MIN_PROMPT_LEN or len(summary) < MIN_SUMMARY_LEN:
            skipped += 1
            continue
        examples.append({
            'messages': [
                {'role': 'user',      'content': prompt},
                {'role': 'assistant', 'content': summary},
            ]
        })

with open(OUT_JSONL, 'w') as f:
    for ex in examples:
        f.write(json.dumps(ex) + '\n')

# ── Statistics ────────────────────────────────────────────────────────────────
prompt_lens  = [len(ex['messages'][0]['content']) for ex in examples]
summary_lens = [len(ex['messages'][1]['content']) for ex in examples]

print(f'{"Collected":<14}: {len(examples)} examples')
print(f'{"Skipped":<14}: {skipped} (too short)')
print(f'{"Written to":<14}: {OUT_JSONL}')
print()
print(f'{"Prompt len":<14}  min={min(prompt_lens):4d}  max={max(prompt_lens):4d}  mean={sum(prompt_lens)/len(prompt_lens):.0f}')
print(f'{"Summary len":<14}  min={min(summary_lens):4d}  max={max(summary_lens):4d}  mean={sum(summary_lens)/len(summary_lens):.0f}')

# ── Length distribution plot ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(prompt_lens,  bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Prompt length (chars)')
axes[0].set_xlabel('Characters')
axes[1].hist(summary_lens, bins=20, color='coral',     edgecolor='white')
axes[1].set_title('Summary length (chars)')
axes[1].set_xlabel('Characters')
plt.tight_layout()
plt.show()

# ── Print first 3 examples ────────────────────────────────────────────────────
N_SHOW = min(3, len(examples))
print(f'\n{"="*70}')
print(f'First {N_SHOW} training examples')
print(f'{"="*70}')
for i, ex in enumerate(examples[:N_SHOW]):
    prompt  = ex['messages'][0]['content']
    summary = ex['messages'][1]['content']
    print(f'\n── Example {i+1} ─────────────────────────────────────────────────')
    print(f'[USER]\n{prompt}')
    print(f'\n[ASSISTANT]\n{summary}')
print(f'\n{"="*70}')

## Section 4 — Fine-tune Gemma 4 2B (QLoRA)

Loads the model in 4-bit NF4, wraps it with LoRA adapters on all attention + MLP
projection layers, then trains with `SFTTrainer`. Loss is logged every 5 steps.
A plot is shown after training completes.

In [ ]:
# Cell 14 — Fine-tuning configuration
MODEL_ID          = 'google/gemma-4-2b-it'
ADAPTER_SAVE_PATH = '/content/drive/MyDrive/Fusion_sensor/ckpt/gemma4_lora'

LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

MAX_SEQ_LEN   = 1024
BATCH_SIZE    = 1
GRAD_ACCUM    = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 3
WARMUP_RATIO  = 0.03

print('Fine-tuning config ready.')

In [ ]:
# Cell 15 — Load tokenizer + model in 4-bit (QLoRA)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False  # required for gradient checkpointing

total     = sum(p.numel() for p in model.parameters())
print(f'Model loaded. Total parameters: {total:,}')

In [ ]:
# Cell 16 — Apply LoRA adapter
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 17 — Train
from datasets import load_dataset as hf_load
from trl import SFTTrainer, SFTConfig

train_dataset = hf_load('json', data_files=str(OUT_JSONL), split='train')
print(f'Training on {len(train_dataset)} examples')

training_args = SFTConfig(
    output_dir='/content/checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    bf16=True,
    logging_steps=5,
    save_strategy='epoch',
    optim='paged_adamw_8bit',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

trainer.train()

# ── Plot training loss ────────────────────────────────────────────────────────
logs   = [x for x in trainer.state.log_history if 'loss' in x]
steps  = [x['step'] for x in logs]
losses = [x['loss'] for x in logs]

plt.figure(figsize=(10, 4))
plt.plot(steps, losses, marker='o', markersize=3, linewidth=1.5, color='steelblue')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final loss: {losses[-1]:.4f}')

In [ ]:
# Cell 18 — Save LoRA adapter to Drive
from pathlib import Path as _Path
_Path(ADAPTER_SAVE_PATH).mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f'Adapter saved to {ADAPTER_SAVE_PATH}')

## Section 5 — Evaluate Fine-tuned Model

Runs the fine-tuned adapter on:
1. A held-out training example (to verify it learned the style)
2. A new prompt it has never seen

In [ ]:
# Cell 19 — Compare training example vs new prompt
model.config.use_cache = True
model.eval()

def generate(prompt_text, max_new_tokens=256):
    messages  = [{'role': 'user', 'content': prompt_text}]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt'
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)


# ── 1. Held-out training example ─────────────────────────────────────────────
if examples:
    sample       = examples[0]['messages']
    known_prompt = sample[0]['content']
    expected     = sample[1]['content']
    predicted    = generate(known_prompt)

    print('=' * 70)
    print('TRAINING EXAMPLE (model has seen this)')
    print('=' * 70)
    print('[PROMPT]')
    print(known_prompt)
    print('\n[EXPECTED]')
    print(expected)
    print('\n[PREDICTED]')
    print(predicted)

# ── 2. Unseen prompt ──────────────────────────────────────────────────────────
NEW_PROMPT = (
    'Scene: 3 active tracks. '
    'Track 1 (car) at depth 18.2 m, moving right. '
    'Track 2 (person) at depth 6.4 m, crossing path. '
    'Track 3 (cyclist) at depth 11.0 m, approaching. '
    'BEV shows high occupancy at z=6–12 m. '
    'Describe the scene and recommend an action.'
)
response = generate(NEW_PROMPT)

print('\n' + '=' * 70)
print('UNSEEN PROMPT')
print('=' * 70)
print('[PROMPT]')
print(NEW_PROMPT)
print('\n[RESPONSE]')
print(response)